In [0]:
%run "../common/define_path"

In [0]:
tracking_raw_path
tracking_br_path

In [0]:
def read_order_data():
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
    from pyspark.sql.functions import current_timestamp
  
    order_schema = StructType([
    StructField("ord",StringType()),
    StructField("odate",StringType()),
    StructField("wt",StringType()),
    StructField("ordervalue",StringType()),
    StructField("productcategory",StringType()),
    StructField("productname",StringType()),
    StructField("productsize",StringType()),
    StructField("productvalue",StringType()),
    StructField("productweight",StringType()),
    StructField("noofitems",StringType()),
    StructField("sendername",StringType()),
    StructField("sendercity",StringType()),
    StructField("senderstate",StringType()),
    StructField("receivername",StringType()),
    StructField("receivercity",StringType()),
    StructField("receiverstate",StringType()),
    StructField("estcost",StringType()),
    StructField("shipno",StringType()),
    StructField("dcost",StringType()),
    StructField("carriername",StringType()),
    StructField("carriercity",StringType()),
    StructField("carrierstate",StringType())
    ])

    raworder_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option('cloudFiles.schemaLocation',f'{tracking_br_path}/schema')
        .option('header','true')
        .schema(order_schema)
        .load(tracking_raw_path+'/order/')
        .withColumn("ingestion_time",current_timestamp())
    )
    return raworder_stream


In [0]:
def write_order_Data(StreamingDF):

    write_Stream = (StreamingDF.writeStream
                    .format('delta')
     .option("checkpointLocation",f'{tracking_br_path}/checkpoint/order/')
    .outputMode('append')
    .queryName('orderwritestream')
    .trigger(availableNow=True)
    .toTable("dev.tracking_br.order"))
    
    write_Stream.awaitTermination()


In [0]:
df_order = read_order_data()
write_order_Data(df_order)
